# Conditional VAE

Train a 128×128 VAE on training images containing pinhole or delamination. The three-value label vector conditions both encoder and decoder.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

sys.path.append(str(Path.cwd()))
from multilabel_utils import LABEL_COLUMNS, get_device, set_seed
from generative_models import ConditionalVAE

SEED = 42
LATENT_DIM = 64
EPOCHS = 30
set_seed(SEED)
device = get_device()
PROJECT_ROOT = Path.cwd().parents[1]
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "multilabel"
IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "archive" / "classification" / "images"
MODEL_DIR = PROJECT_ROOT / "models" / "multilabel"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
train_df = train_df[(train_df["Delamination"] == 1) | (train_df["Pinhole"] == 1)].copy()
val_df = val_df[(val_df["Delamination"] == 1) | (val_df["Pinhole"] == 1)].copy()
print("Generative training images:", len(train_df))
print("Generative validation images:", len(val_df))

Generative training images: 483
Generative validation images: 98


In [2]:
image_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

class GenerativeDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = Image.open(IMAGE_DIR / row["file_name"]).convert("RGB")
        image = image_transform(image)
        condition = torch.tensor(
            row[LABEL_COLUMNS].to_numpy(dtype="float32"),
            dtype=torch.float32,
        )
        return image, condition

train_loader = DataLoader(
    GenerativeDataset(train_df), batch_size=16, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(GenerativeDataset(val_df), batch_size=16)

In [3]:
model = ConditionalVAE(LATENT_DIM, len(LABEL_COLUMNS)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
BETA = 0.01

def vae_loss(reconstructed, images, mu, logvar):
    batch_size = images.size(0)
    reconstruction = F.mse_loss(
        reconstructed, images, reduction="sum"
    ) / batch_size
    kl_divergence = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    ) / batch_size
    return reconstruction + BETA * kl_divergence

def run_epoch(loader, training):
    model.train() if training else model.eval()
    total_loss = 0.0

    for images, conditions in loader:
        images = images.to(device)
        conditions = conditions.to(device)
        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            reconstructed, mu, logvar = model(images, conditions)
            loss = vae_loss(reconstructed, images, mu, logvar)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)

In [4]:
checkpoint_path = MODEL_DIR / "conditional_vae_best.pth"
best_validation_loss = float("inf")

for epoch in range(EPOCHS):
    train_loss = run_epoch(train_loader, training=True)
    validation_loss = run_epoch(val_loader, training=False)

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        torch.save(model.state_dict(), checkpoint_path)

    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Train: {train_loss:.2f} | Validation: {validation_loss:.2f}"
        )

print("Saved:", checkpoint_path)

Epoch 1/30 | Train: 558.56 | Validation: 499.85


Epoch 5/30 | Train: 149.58 | Validation: 152.83


Epoch 10/30 | Train: 97.08 | Validation: 101.01


Epoch 15/30 | Train: 74.63 | Validation: 83.80


Epoch 20/30 | Train: 70.50 | Validation: 78.82


Epoch 25/30 | Train: 59.56 | Validation: 80.61


Epoch 30/30 | Train: 46.34 | Validation: 94.19
Saved: /Users/kevinnishitoyo/Desktop/Battery Electrode Defect Augmentation/battery-electrode-defect-augmentation/models/multilabel/conditional_vae_best.pth


## Limitation

Pixel reconstruction losses can prioritize the large coating background over small defects. Generated images must be inspected and evaluated downstream.